# 01 · Inspección de predicciones

Lectura cualitativa de `runs/*/predictions.jsonl`: comparar modelos sobre las mismas imágenes y detectar patrones de error (formato vs. contenido) antes de cerrar el extractor de atributos. Las métricas definitivas se calculan en `src/vlmfid/eval`, no aquí.

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from vlmfid.data import data_root
from vlmfid.paths import RUNS

In [ ]:
def load_run(exp_id):
    with open(RUNS / exp_id / "predictions.jsonl") as f:
        return pd.DataFrame([json.loads(l) for l in f])

runs = sorted(p.parent.name for p in RUNS.glob("*/predictions.jsonl"))
status = {r: json.loads((RUNS / r / "status.json").read_text()) for r in runs}
pd.DataFrame(status).T[["status", "n_done", "n_total", "img_per_s", "peak_mem_gb"]]

In [ ]:
preds = pd.concat([load_run(r) for r in runs], ignore_index=True)
wide = preds.pivot_table(index="id", columns="exp_id", values="prediction", aggfunc="first")
wide.insert(0, "reference", preds.groupby("id").reference.first())
common = wide.dropna()
print(f"{len(common)} imágenes con predicción en todos los runs")

In [ ]:
paths = preds.groupby("id").image_path.first()
roots = preds.groupby("id").variant.first().map(data_root)

def show(i):
    r = common.iloc[i]
    plt.figure(figsize=(3, 3)); plt.imshow(Image.open(roots[r.name] / paths[r.name])); plt.axis("off"); plt.show()
    print("REF :", r.reference)
    for c in common.columns[1:]:
        print(f"{c[:45]:<45} {r[c]}")

show(0)

## Longitud de las respuestas por modelo y prompt
Respuestas mucho más largas que la referencia penalizan BLEU/CIDEr aunque el contenido sea correcto: es la señal del *desalineamiento de formato* que la etapa 2 debería corregir.

In [ ]:
preds["n_words"] = preds.prediction.str.split().str.len()
preds.groupby(["model", "prompt_id"]).n_words.describe()[["mean", "50%", "max"]]

## VL-JEPA: prototipos por atributo
Exactitud *provisional* de la predicción por prototipos (no pasa por texto).

In [ ]:
vj = preds[preds.model.str.startswith("vljepa")]
if len(vj):
    attrs = list(vj.extra.iloc[0]["attr_pred_proto"])
    acc = {f: (vj.extra.map(lambda e: e["attr_pred_proto"][f]) == vj.latents_text.map(lambda z: z[f])).groupby(vj.exp_id).mean()
           for f in attrs}
    display(pd.DataFrame(acc))